In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from iteration_function import mandelbrot

from scipy.integrate import solve_ivp

### Question 1

In [ ]:
x, y, bounded, divergence_iter = mandelbrot(N=1000, max_iter=100)  # calling mandelbrot function from mandelbrot.py

In [ ]:
# plotting
plt.figure(figsize=(10, 10))
plt.imshow(bounded, extent=[-2, 2, -2, 2])
plt.xlabel(r"$\Re(c)$", fontsize=28)
plt.ylabel(r"$\Im(c)$", fontsize=28)
plt.xticks(fontsize=19)
plt.yticks(fontsize=19)
plt.savefig("Question_1a_plot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# plotting
plt.figure(figsize=(12, 12))

im = plt.imshow(divergence_iter, extent=[-2, 2, -2, 2])
cbar = plt.colorbar(im, shrink=0.82)
cbar.set_label("Iteration number at divergence", fontsize=25)
cbar.ax.tick_params(labelsize=19)

plt.xlabel(r"$\Re(c)$", fontsize=28)
plt.ylabel(r"$\Im(c)$", fontsize=28)
plt.xticks(fontsize=19)
plt.yticks(fontsize=19)
plt.savefig("Question_1b_plot.pdf", bbox_inches="tight")
plt.show()

### Question 2

In [ ]:
def lorenz(t, W, sigma=10, r=28, b=8/3):
    """
    Function computes the time derivatives for the Lorenz system

    Parameters:
    t : current time (float)
    W : current state vector W[X, Y, Z] (array-like)
    sigma : Prandtl number (float)
    r : Rayleigh number (float)
    b : Dimensionless length scale (float)

    Returns:
    dX/dt, dY/dt, dZ/dt : time derivatives for the Lorenz system
    """
    X, Y, Z = W  # unpacking state vector

    dXdt = -sigma * (X - Y)  # Lorenz system
    dYdt = r * X - Y - X * Z
    dZdt = -b * Z + X * Y

    return [dXdt, dYdt, dZdt]

sigma, r, b = 10, 28, 8/3  # setting Lorenz parameters
W0 = [0, 1, 0]  # setting initial conditions
dt = 0.01  # setting time spacing between output points
t_end = 60  # setting final integration time
t_eval = np.arange(0, t_end + dt, dt)  # array of times: 0, 0.01, 0.02, ..., 60

sol = solve_ivp(lorenz, t_span=(0, t_end), y0=W0, t_eval=t_eval, args=(sigma, r, b))  # calling ODE solver on lorenz

X, Y, Z = sol.y  # extracting solution arrays
t = sol.t  # extracting time array from solution
N = t / dt  # converting time into iteration number

In [ ]:
# plotting; reproducing Lorenz Figure 1
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharey=True)

for i, ax in enumerate(axes):  # looping over subplots
    start = i * 1000  # setting starting iteration number for current panel
    end = (i + 1) * 1000  # setting ending iteration number for current panel

    ax.plot(N[start:end], Y[start:end], color="#0018FF")  # plotting Y versus iteration number N
    ax.axhline(0, color="red", linestyle="--")
    ax.set_xlim(start, end)  # setting horizontal axis limits
    ax.set_ylabel(r"$Y$", fontsize=25)
    ax.tick_params(axis='both', labelsize=16)

axes[-1].set_xlabel(r"$N = t/\Delta t$", fontsize=25)

plt.tight_layout()
plt.savefig("Question_2a_plot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
mask = (N >= 1400) & (N <= 1900)  # Boolean mask selecting relevant N-range from Lorenz

fig, axes = plt.subplots(2, 1, figsize=(9, 10))

axes[0].plot(Y[mask], Z[mask], color='#C11BCF', linewidth=1.5)
axes[0].set_xlabel(r"$Y$", fontsize=25)
axes[0].set_ylabel(r"$Z$", fontsize=25)
axes[0].tick_params(axis="both", labelsize=16)

axes[1].plot(Y[mask], X[mask], color='#C11BCF', linewidth=1.5)
axes[1].set_xlabel(r"$Y$", fontsize=25)
axes[1].set_ylabel(r"$X$", fontsize=25)
axes[1].tick_params(axis="both", labelsize=16)

for n in [1400, 1500, 1600, 1700, 1800, 1900]:
    idx = np.argmin(np.abs(N - n))  # finding where array N is closest to value of n (for labeling iterations)

    axes[0].text(Y[idx], Z[idx], str(n // 100), fontsize=15)
    axes[1].text(Y[idx], X[idx], str(n // 100), fontsize=15)

plt.tight_layout()
plt.savefig("Question_2b_plot.pdf", bbox_inches="tight")
plt.show()

In [ ]:
W0_prime = [0, 1.00000001, 0]  # # setting perturbed initial conditions

# calling ODE solver on lorenz
sol_prime = solve_ivp(lorenz, t_span=(0, t_end), y0=W0_prime, t_eval=t_eval, args=(sigma, r, b))

X_prime, Y_prime, Z_prime = sol_prime.y  # extracting solution arrays

# computing distance between original trajectory W(t) and perturbed trajectory W'(t) at each time
distance = np.sqrt((X_prime - X)**2 + (Y_prime - Y)**2 + (Z_prime - Z)**2)

# plotting
plt.figure(figsize=(9, 7))
plt.semilogy(t, distance, color="#BD7A08")  # plotting distance vs. time using logarithmic vertical axis (semilogy)
plt.xlabel(r"$t$", fontsize=25)
plt.ylabel(r"$d(t)$", fontsize=25)
plt.tick_params(axis="both", labelsize=16)
plt.savefig("Question_2c_plot.pdf", bbox_inches="tight")
plt.show()